# DriveSense-VLM — 03: Inference Benchmarks

**GPU**: A100 80GB | **Time**: ~1 h | **Cost**: ~12 CU

Measures production inference performance:
- **vLLM backend** — concurrent throughput (AWQ 4-bit, A100)
- **Local transformers backend** — sequential latency (T4-equivalent)
- **ViT-only** — TensorRT vs eager vs torch.compile

| Target | Threshold |
|--------|-----------|
| E2E latency A100 p50 | < 200 ms |
| E2E latency T4 p50 | < 500 ms |
| ViT p50 | < 25 ms |
| Throughput A100 | ≥ 8 fps |

> ⚠️ **Before running**: Runtime → Change runtime type → **A100 GPU**
>
> **Prerequisites**: `02_optimization.ipynb` must have produced the quantized model.

In [ ]:
# ── Mount Google Drive ──
from google.colab import drive
drive.mount('/content/drive')

import os, sys

# ── Paths ──
PROJECT_ROOT = "/content/drive/MyDrive/DriveSense-VLM"
REPO_ROOT    = "/content/DriveSense-VLM"
DATA_ROOT    = f"{PROJECT_ROOT}/data"
OUTPUTS_ROOT = f"{PROJECT_ROOT}/outputs"

# Create Drive directories
for d in [DATA_ROOT, f"{DATA_ROOT}/nuscenes", f"{DATA_ROOT}/dada2000",
          OUTPUTS_ROOT, f"{OUTPUTS_ROOT}/data", f"{OUTPUTS_ROOT}/training",
          f"{OUTPUTS_ROOT}/data/sft_ready"]:
    os.makedirs(d, exist_ok=True)

# Clone repo to fast ephemeral SSD
if not os.path.exists(REPO_ROOT):
    !git clone https://github.com/jayanth922/DriveSense-VLM.git {REPO_ROOT}
else:
    !git -C {REPO_ROOT} pull --quiet
os.chdir(REPO_ROOT)

# Symlink data/outputs → Drive (persistent across disconnects)
!ln -sfn {DATA_ROOT} {REPO_ROOT}/data
!ln -sfn {OUTPUTS_ROOT} {REPO_ROOT}/outputs

# Add src to Python path (replaces broken editable install)
sys.path.insert(0, f"{REPO_ROOT}/src")
print(f"✓ Repo:  {REPO_ROOT}")
print(f"✓ Drive: {PROJECT_ROOT}")

In [ ]:
# Note: nuscenes-devkit (used in nb00/05) must be installed with --no-deps
#       to skip its strict numpy version pin. Not needed in this notebook.
# Install benchmark dependencies
!pip install --upgrade pip setuptools wheel -q 2>&1 | tail -1
!pip install pyyaml tqdm Pillow requests -q 2>&1 | tail -1
!pip install transformers peft accelerate -q 2>&1 | tail -1
!pip install vllm -q 2>&1 | tail -3

# Verify GPU
import torch
assert torch.cuda.is_available(), "No GPU — set Runtime → A100 GPU"
print(f"✓ GPU : {torch.cuda.get_device_name(0)}")
print(f"✓ VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

import drivesense
print(f"✓ drivesense importable")

In [ ]:
# Run vLLM benchmark (concurrent throughput, A100)
import os

BENCH_DIR = f"{OUTPUTS_ROOT}/benchmarks"
QUANT_DIR = f"{OUTPUTS_ROOT}/quantized_model"
os.makedirs(BENCH_DIR, exist_ok=True)

MOCK_FLAG = "" if os.path.exists(QUANT_DIR) else "--mock"
if MOCK_FLAG:
    print("⚠ Quantized model not found — running in mock mode.")

!python scripts/run_benchmark.py \
    --vllm \
    --output {BENCH_DIR}/vllm_bench.json \
    {MOCK_FLAG} \
    2>&1

In [ ]:
# Run local (transformers) benchmark — simulates T4 deployment
import os

BENCH_DIR = f"{OUTPUTS_ROOT}/benchmarks"
MOCK_FLAG = "" if os.path.exists(f"{OUTPUTS_ROOT}/quantized_model") else "--mock"

!python scripts/run_benchmark.py \
    --local \
    --output {BENCH_DIR}/local_bench.json \
    {MOCK_FLAG} \
    2>&1

# Run ViT-only benchmark (TensorRT vs alternatives)
!python scripts/run_benchmark.py \
    --vit-only \
    --output {BENCH_DIR}/vit_bench.json \
    {MOCK_FLAG} \
    2>&1

In [ ]:
# Display benchmark results table
import os, json

BENCH_DIR = f"{OUTPUTS_ROOT}/benchmarks"

def load(fname):
    p = f"{BENCH_DIR}/{fname}"
    return json.loads(open(p).read()) if os.path.exists(p) else {}

vllm  = load("vllm_bench.json")
local = load("local_bench.json")
vit   = load("vit_bench.json")

targets = {
    "A100 p50 (ms)": (vllm.get("p50_ms"),   200,  "<"),
    "T4 p50 (ms)":   (local.get("p50_ms"),  500,  "<"),
    "ViT p50 (ms)":  (vit.get("p50_ms"),     25,  "<"),
    "Throughput fps":(vllm.get("fps"),         8,  "≥"),
}

print("=" * 55)
print("  Benchmark Results")
print("=" * 55)
for label, (val, tgt, op) in targets.items():
    if val is None:
        status = "-"
        val_str = "N/A"
    else:
        passes = (val < tgt) if op == "<" else (val >= tgt)
        status = "✓" if passes else "✗"
        val_str = f"{val:.1f}"
    print(f"  {status}  {label:<22}: {val_str:<8} target {op}{tgt}")
print("=" * 55)
print("\nFull data saved to:", BENCH_DIR)
print("Proceed to 04_evaluation.ipynb")